# Filler in LLM answers: synthetic reports (T4)
Runtime → Change runtime type → **T4 GPU**, then run the cell below. Settings are at the top of the cell; rerunning resumes where it stopped.

## Optional: restore a saved run
In a new Colab session, run this cell and pick the `.zip` that the Details cell downloaded (e.g. `t4_n20_v12.zip`); it goes to Google Drive, and the main cell then reuses its responses instead of generating them again.

In [ ]:
# Restore a run folder saved by the Details cell (skip in the session that made it).
TO_DRIVE = True  # False: into /content (lost when the runtime is recycled; set USE_DRIVE = False too)
import zipfile
from pathlib import Path
from google.colab import files

if TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/drybench_runs" if TO_DRIVE else "/content/drybench_runs")
dest.mkdir(parents=True, exist_ok=True)
for name in files.upload():
    zipfile.ZipFile(name).extractall(dest)
    print("restored", name, "→", dest)


In [ ]:
# Filler in LLM answers, synthetic reports — full run on a Colab T4 GPU
# 1) Runtime → Change runtime type → T4 GPU.   2) Run this cell (~55 min with the defaults:
#    300 items × 3 models × 2 prompts; N_PER_CELL = 6 takes ~20 min).
# Re-running the cell resumes: every step appends to JSONL files and skips finished rows.
# A newer version of this cell on an old run rescores it and reuses the generated responses.
# The drybench package (v1.2.0) is embedded at the bottom of the cell; nothing to upload.

# ---- settings ----
N_PER_CELL = 20                      # items per knob cell; 15 cells → 300 items
MODELS = {                           # label: Hugging Face id of an evaluated (instruct) model
    "qwen2.5-0.5b": "Qwen/Qwen2.5-0.5B-Instruct",
    "smollm2-1.7b": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "qwen2.5-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    # "phi3.5-mini": "microsoft/Phi-3.5-mini-instruct",  # optional 4th model (3.8B), ~+25 min
}
READER = "Qwen/Qwen2.5-3B-Instruct"  # reads the responses; keep it out of MODELS
VARIANTS = ["default", "concise"]    # "concise" adds "Be concise." to the system prompt (H3)
FILLER_CHECK_N = 3                    # simulated items per cell to test the filler map with this reader (0 = skip)
MAX_NEW_TOKENS = 320
SCORE_MODE = "loo"                   # "greedy" is exact but several times slower
SAMPLED = 0                          # >0: also sampled answer entropy from N samples (slower)
DTYPE = "fp16"                       # T4 has no bf16; use "fp32" if a step reports NaN logits
BATCH = 16                           # generation batch size
USE_DRIVE = True                     # keep outputs in Google Drive (survives disconnects); False: /content
RUN_ROOT = "/content/drybench_runs"
RUN_NAME = "t4_n20_v12"              # new name = fresh run; same name = resume
ALLOW_CPU = False
# ---- end settings ----

import base64
import hashlib
import io
import json
import os
import re
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

ENV = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
PROGRESS = re.compile(r"^(?:generated|signals|scored) (\d+)/(\d+)")


def run(args, check=True):
    args = [str(a) for a in args]
    shown = ["drybench"] + args[3:] if args[1:3] == ["-m", "drybench.cli"] else args
    print("$", " ".join(shown), flush=True)
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENV)
    for line in p.stdout:
        if "it/s]" in line or "s/it]" in line:
            continue
        m = PROGRESS.match(line.strip())  # keep about ten progress lines per step
        if m and int(m.group(1)) % max(10, int(m.group(2)) // 10) and m.group(1) != m.group(2):
            continue
        print("   ", line, end="", flush=True)
    code = p.wait()
    print(f"    [{time.time() - t0:.0f}s]", flush=True)
    if check and code != 0:
        raise RuntimeError(f"step failed with exit code {code}: {' '.join(args)}")
    return code


def install():
    digest = hashlib.sha1(PKG_B64.encode()).hexdigest()[:10]
    base = Path("/content") if Path("/content").exists() else Path.cwd()
    pkg = (base / f"drybench_pkg_{digest}").resolve()
    marker = pkg / ".installed"
    if not marker.exists():
        zipfile.ZipFile(io.BytesIO(base64.b64decode(PKG_B64))).extractall(pkg)
        ENV["PIP_BREAK_SYSTEM_PACKAGES"] = "1"  # harmless on Colab; needed on PEP 668 systems
        run([sys.executable, "-m", "pip", "install", "-q", str(pkg), "transformers>=4.46", "accelerate",
             "matplotlib"])
        marker.touch()
    return pkg


def hf_token():
    try:
        from google.colab import userdata

        tok = userdata.get("HF_TOKEN")
        if tok:
            ENV["HF_TOKEN"] = tok
    except Exception:
        pass  # public models do not need a token


def num(x, spec="+.3f"):
    return "n/a" if x != x else format(x, spec)  # x != x is True only for NaN


def fmt_ci(p, s, spec="+.3f"):
    return f"{num(p[s], spec)} [{num(p[s + '_lo'], spec)}, {num(p[s + '_hi'], spec)}]"


def verdict(p, s, up, down, flat):
    lo, hi = p[s + "_lo"], p[s + "_hi"]
    if lo != lo or hi != hi:
        return "n/a"
    return up if lo > 0 else (down if hi < 0 else flat)


def paired_ci(x, ratio=False, n_boot=2000, seed=0):
    """Mean of paired differences (or geometric mean of ratios) with a bootstrap 95% CI."""
    import numpy as np

    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return "n/a"
    boots = np.random.default_rng(seed).choice(x, (n_boot, len(x))).mean(axis=1)
    m, lo, hi = x.mean(), *np.quantile(boots, [0.025, 0.975])
    if ratio:
        return f"×{np.exp(m):.2f} [{np.exp(lo):.2f}, {np.exp(hi):.2f}]"
    return f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]"


def show_results(out, report):
    import numpy as np
    import pandas as pd

    pd.set_option("display.width", 200)
    rc = json.loads((out / "reader_check.json").read_text())
    ok_reader = rc["auc_present_vs_absent"] >= 0.95 and rc["reader_fail_rate"] <= 0.1
    print("\n================ RESULTS ================")
    print(f"Reader check: AUC present vs absent = {rc['auc_present_vs_absent']:.3f}, "
          f"cannot read gold answer in {100 * rc['reader_fail_rate']:.0f}% of items "
          f"→ {'reader OK' if ok_reader else 'WEAK READER: treat W with caution'}")
    active = rc.get("misleading_check_active", float("nan"))
    if active == active:
        print(f"Wrong-statement detection works for {100 * active:.0f}% of questions "
              f"(reader answers 'unknown' to an empty message with median p = "
              f"{rc['p_unknown_empty_median']:.2f}).")
    else:
        print("Wrong-statement detection is off: the reader has no chat template.")

    wc = out / "filler_check" / "summary.json"
    if wc.exists():
        ws = json.loads(wc.read_text())
        names = {"loo": "tolerance 0.5 nats (default)", "loo_t1": "tolerance 1 nat",
                 "loo_t2": "tolerance 2 nats", "loo_global": "one budget for all facts"}
        tab = pd.DataFrame({lab: {"no fact lost": ws[k]["safety_all_ok"], "filler precision": ws[k]["filler_precision"],
                                  "filler recall": ws[k]["filler_recall"], "MAE vs true W": ws[k]["mae"]}
                            for k, lab in names.items() if k in ws}).T
        print(f"\nFiller-map check: {ws['n_responses']} simulated responses with known filler, read by this reader:")
        print(tab.round(3).to_string())
        kept = {k.replace("filler:", ""): v for k, v in ws["loo"]["kept_share_by_label"].items()
                if k.startswith("filler:") and v > 0.05}
        if kept:
            print("  filler the default setting leaves in: " + ", ".join(
                f"{k} {100 * v:.0f}%" for k, v in sorted(kept.items(), key=lambda kv: -kv[1])))

    groups = pd.read_csv(report / "groups.csv", index_col=0)
    print("\nPer model × prompt:\n"
          "  W = filler share of tokens (uninformative + repeated facts); W_wrong = wrong or confusing statements "
          "(errors, not filler);\n"
          "  κ = share of the requested information delivered; facts = share of requested facts the reader "
          "gets right; bits/tok = density;\n"
          "  overhead = tokens per delivered bit relative to the minimal correct answer (1.0 = as lean)\n"
          "           = ×removable (sentences that can go) × ×wordiness (kept tokens per delivered bit vs the gold answer)")
    cols = {"n": "n", "share_ok": "scored", "tokens_mean": "tokens", "W_mean": "W",
            "W_misleading_mean": "W_wrong", "kappa_mean": "κ", "facts_mean": "facts",
            "density_mean": "bits/tok", "overhead_gmean": "overhead", "removable_gmean": "×removable",
            "wordiness_gmean": "×wordiness"}
    print(groups[[c for c in cols if c in groups]].rename(columns=cols).round(3).to_string())
    if "W_ablation_mean" in groups and groups["W_ablation_mean"].notna().any():
        diff = (groups["W_ablation_mean"] - groups["W_mean"]).abs().max()
        print(f"Tolerance check: with a looser tolerance (2 nats instead of 0.5) mean W changes by at most "
              f"{diff:.3f} across groups.")

    prof_path = report / "profiles.csv"
    prof = pd.read_csv(prof_path, index_col=0) if prof_path.exists() and prof_path.stat().st_size > 5 else None
    if prof is None or prof.empty:
        print("\nNo profiles: too few scored responses with entropy signals.")
        return
    print("\nProfiles (point [95% CI]). α = filler on easy items the model is sure about; "
          "β = filler vs uncertainty (β_ctrl: within the same item design);\n"
          "γ, γ_filler = substance / filler tokens per requested fact (plain items):")
    for g, p in prof.iterrows():
        print(f"  {g}: n={int(p['n'])}  α={num(p['alpha'], '.3f')} (n={int(p['alpha_n'])})  "
              f"β={fmt_ci(p, 'beta')}  β_ctrl={fmt_ci(p, 'beta_ctrl')}\n"
              f"  {' ' * len(g)}  γ={fmt_ci(p, 'gamma', '+.2f')}  γ_filler={fmt_ci(p, 'gamma_filler', '+.2f')}")

    print("\nH2 — does filler follow uncertainty while substance follows the number of facts?")
    for g, p in prof.iterrows():
        print(f"  {g}: filler vs uncertainty: {verdict(p, 'beta', 'grows', 'falls', 'no significant trend')} "
              f"(same design: {verdict(p, 'beta_ctrl', 'grows', 'falls', 'no significant trend')}); "
              f"substance vs k: {verdict(p, 'gamma', 'grows', 'falls', 'no significant trend')}; "
              f"filler vs k: {verdict(p, 'gamma_filler', 'grows', 'falls', 'no significant trend')}")

    it = pd.read_csv(report / "item_table.csv")
    if {"default", "concise"} <= set(VARIANTS):
        print("\nH3 — 'Be concise.' vs default on the same items (concise − default, mean [95% CI]; "
              "overhead as a ratio):")
        piv = it.pivot_table(index=["model", "item_id"], columns="variant", aggfunc="first", dropna=False,
                             values=["tokens", "W", "W_misleading", "kappa", "facts", "overhead"])
        rows = {}
        for m, g in piv.groupby(level=0):
            def d(c, op=np.subtract):
                if (c, "concise") not in g.columns or (c, "default") not in g.columns:
                    return np.array([])
                return op(g[(c, "concise")].to_numpy(float), g[(c, "default")].to_numpy(float))

            with np.errstate(divide="ignore", invalid="ignore"):
                log_ratio = np.log(d("overhead", np.divide))
            rows[m] = {"Δtokens": paired_ci(d("tokens")), "ΔW": paired_ci(d("W")),
                       "ΔW_wrong": paired_ci(d("W_misleading")), "Δκ": paired_ci(d("kappa")),
                       "Δfacts": paired_ci(d("facts")), "overhead": paired_ci(log_ratio, ratio=True)}
        print(pd.DataFrame(rows).T.to_string())

    att_path = report / "attribution.csv"
    if att_path.exists():
        att = pd.read_csv(att_path, index_col=0)
        ws_path = report / "filler_spans.csv"
        if ws_path.exists():
            att["filler spans"] = pd.read_csv(ws_path).groupby("group").size()
        print("\nH1 — share of filler tokens by entropy cell:")
        print(att.round(3).to_string())
        enr_path = report / "attribution_enrichment.csv"
        if enr_path.exists():
            print("    the same, relative to the share of ALL response tokens in the cell "
                  "(> 1: filler is over-represented there):")
            print(pd.read_csv(enr_path, index_col=0).round(2).to_string())

    try:
        from IPython.display import Image, display

        for f in ("fig_filler_vs_entropy.png", "fig_tokens_vs_k.png", "fig_attribution.png"):
            if (report / f).exists():
                display(Image(filename=str(report / f)))
    except ImportError:
        pass
    print(f"\nAll tables and figures: {report}")


def main():
    t_start = time.time()
    try:
        import torch

        gpu = torch.cuda.is_available()
        print("GPU:", torch.cuda.get_device_name(0) if gpu else "none")
    except ImportError:
        gpu = False
    if not gpu and not ALLOW_CPU:
        raise SystemExit("No GPU. In Colab: Runtime → Change runtime type → T4 GPU, then rerun.")
    dtype = DTYPE if gpu else "auto"
    if USE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")
        root = Path("/content/drive/MyDrive/drybench_runs")
    else:
        root = Path(RUN_ROOT)
    out = root / RUN_NAME
    out.mkdir(parents=True, exist_ok=True)
    print("Outputs:", out)
    pkg = install()
    hf_token()
    cli = [sys.executable, "-m", "drybench.cli"]
    items, resp = out / "items.jsonl", out / "responses.jsonl"
    signals, scored, report = out / "signals.jsonl", out / "scored.jsonl", out / "report"

    # Scoring or signals changed since these outputs were made? Recompute them, keep responses.
    got = subprocess.run([sys.executable, "-c", "import drybench, json; print(json.dumps("
                          "{'scoring': drybench.SCORING_VERSION, 'signals': drybench.SIGNALS_VERSION}))"],
                         capture_output=True, text=True, env=ENV).stdout.strip()
    if not got:
        raise RuntimeError("drybench is not importable after installation; rerun the cell")
    new = json.loads(got)
    vfile, legacy = out / "versions.json", out / "scoring_version.txt"
    if vfile.exists():
        old = json.loads(vfile.read_text())
    else:  # runs made before versions.json: signals last changed with scoring version 2
        s_old = legacy.read_text().strip() if legacy.exists() else "1"
        old = {"scoring": s_old, "signals": "2" if int(s_old) >= 2 else "1"}
    if old["scoring"] != new["scoring"] and scored.exists():
        # every score, or only those the new version changed for this reader (drybench.rescore)
        got = subprocess.run([str(a) for a in cli + ["prune-scores", "--dir", out, "--items", items, "--reader",
                                                     READER, "--from", old["scoring"], "--json"]],
                             capture_output=True, text=True, env=ENV)
        if got.returncode:
            raise RuntimeError("prune-scores failed:\n" + got.stdout + got.stderr)
        res = json.loads(got.stdout.strip().splitlines()[-1])
        if res["plan"] == "all":
            print(f"Scoring changed (v{old['scoring']} → v{new['scoring']}): updating answer forms of the "
                  "items, recomputing the reader checks and filler maps; responses and signals are reused.")
            if items.exists():
                run(cli + ["items", "--out", items, "--update-probes"])
        else:
            what = ("the answers to items with a missing fact" if res["plan"] == "missing"
                    else "nothing for this reader")
            print(f"Scoring changed (v{old['scoring']} → v{new['scoring']}): rescoring {what}; "
                  "everything else is kept.")
        for line in res["changes"]:
            print("    " + line)
    if old["signals"] != new["signals"] and signals.exists():
        print(f"Signals changed (v{old['signals']} → v{new['signals']}): recomputing entropy signals.")
        signals.unlink()
        shutil.rmtree(report, ignore_errors=True)
    vfile.write_text(json.dumps(new))
    legacy.unlink(missing_ok=True)

    print("\n[1/6] items")
    if not items.exists():
        run(cli + ["items", "--out", items, "--n-per-cell", N_PER_CELL])
    n_items = sum(1 for _ in open(items))
    print(f"    {n_items} items")

    print("\n[2/6] reader check")
    if not (out / "reader_check.json").exists():
        run(cli + ["check-reader", "--items", items, "--reader-backend", "hf", "--reader", READER,
                   "--reader-dtype", dtype, "--n", 30, "--out", out / "reader_check.json"])

    print("\n[3/6] filler-map check on simulated responses with known filler")
    wc = out / "filler_check"
    if FILLER_CHECK_N and not (wc / "summary.json").exists():
        run([sys.executable, pkg / "scripts" / "pilot_validation.py", "--reader-backend", "hf",
             "--reader", READER, "--reader-dtype", dtype, "--out-dir", wc, "--n-per-cell", FILLER_CHECK_N,
             "--greedy-conditions", "none", "--no-plots", "--quiet"])

    print("\n[4/6] generate responses + entropy signals")
    for label, model_id in MODELS.items():
        run(cli + ["generate", "--items", items, "--out", resp, "--backend", "hf", "--model", model_id,
                   "--model-name", label, "--variant", *VARIANTS, "--dtype", dtype,
                   "--batch-size", BATCH, "--max-new-tokens", MAX_NEW_TOKENS])
        run(cli + ["signals", "--items", items, "--responses", resp, "--out", signals, "--backend", "hf",
                   "--model", model_id, "--model-name", label, "--dtype", dtype, "--sampled", SAMPLED])

    print("\n[5/6] filler maps (reader scoring)")
    run(cli + ["score", "--items", items, "--responses", resp, "--out", scored, "--reader-backend", "hf",
               "--reader", READER, "--reader-dtype", dtype, "--reader-max-batch-tokens", 4096,
               "--mode", SCORE_MODE, "--ablation-abs-tol", 2.0])

    print("\n[6/6] analysis")
    run(cli + ["analyze", "--items", items, "--scored", scored, "--signals", signals,
               "--out-dir", report, "--n-boot", 500])
    show_results(out, report)
    print(f"\nTotal time: {(time.time() - t_start) / 60:.1f} min")
    return out


PKG_B64 = "..."

RUN_DIR = str(main())  # the cells below find the run here


## Details
CPU only, after the main cell: tolerance check, filler by item design, span entropy as a filler detector, completeness check, example spans. At the end it zips the run folder and downloads it.

In [ ]:
# Filler in LLM answers — details (CPU only, ~30 s). Run in a new cell after the main cell has finished;
# at the end it zips the run folder and downloads it.
import json
import shutil

import numpy as np
import pandas as pd
from pathlib import Path

RUN = "auto"  # the main cell's folder (Google Drive or /content), or a path


def find_run(run, name="t4_n20_v12"):
    """RUN = "auto": the main cell's folder in this session (RUN_DIR), else the run in Google Drive,
    else the one in /content; a path is used as given."""
    if run != "auto":
        return Path(run)
    if "RUN_DIR" in globals():
        return Path(globals()["RUN_DIR"])
    name = globals().get("RUN_NAME", name)
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception:
            pass  # not in Colab, or Drive declined: look in /content
    on_drive = Path("/content/drive/MyDrive/drybench_runs") / name
    return on_drive if on_drive.exists() else Path("/content/drybench_runs") / name


RUN = find_run(RUN)
print("Run:", RUN)
R = RUN / "report"
it = pd.read_csv(R / "item_table.csv")
sp = pd.read_csv(R / "span_table.csv")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
it["group"] = it["model"] + " | " + it["variant"]
sp["group"] = sp["model"] + " | " + sp["variant"]
sc = it[it["status"] == "ok"].copy()
rng = np.random.default_rng(0)


def ci(x, n=2000):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return "n/a"
    b = rng.choice(x, (n, len(x))).mean(axis=1)
    return f"{x.mean():+.3f} [{np.quantile(b, 0.025):+.3f}, {np.quantile(b, 0.975):+.3f}]"


def auc(pos, neg):
    """P(score of a positive > score of a negative), ties count 1/2."""
    pos, neg = np.asarray(pos, float), np.asarray(neg, float)
    pos, neg = pos[np.isfinite(pos)], neg[np.isfinite(neg)]
    if not len(pos) or not len(neg):
        return float("nan")
    r = pd.Series(np.concatenate([pos, neg])).rank().to_numpy()
    return float((r[: len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg)))


first = json.loads(open(RUN / "scored.jsonl").readline())
ab = first.get("ablation") or {}
alt = f"tolerance {ab.get('abs_tol')} nats" if ab.get("abs_tol") not in (None, first.get("abs_tol")) \
    else f"δ = {ab.get('delta')}"
print(f"1) Filler under two tolerances: {first.get('abs_tol')} nats (default) vs {alt}")
rows = {}
for g, d in sc.groupby("group"):
    rows[g] = {"W": d["W"].mean(), "W (alternative)": d["W_ablation"].mean(),
               "responses changed": ((d["W_ablation"] - d["W"]).abs() > 1e-9).mean()}
print(pd.DataFrame(rows).T.round(3).to_string())
piv = it.pivot_table(index=["model", "item_id"], columns="variant", values=["W", "W_ablation"],
                     aggfunc="first", dropna=False)
print("   H3 ΔW (concise − default) under each tolerance:")
for m, g in piv.groupby(level=0):
    print(f"   {m}: default {ci(g[('W', 'concise')] - g[('W', 'default')])}; "
          f"alternative {ci(g[('W_ablation', 'concise')] - g[('W_ablation', 'default')])}")
wc = RUN / "filler_check" / "summary.json"
if wc.exists():
    ws = json.loads(wc.read_text())
    print("   simulated filler left in (share of spans of each type) by tolerance:")
    kept = pd.DataFrame({name: ws[k]["kept_share_by_label"]
                         for k, name in (("loo", "0.5 nats"), ("loo_t1", "1 nat"), ("loo_t2", "2 nats"))
                         if k in ws})
    print(kept.round(2).to_string())

print("\n2) Where the filler is, by item design (mean W; W_wrong below):")


def design(r):
    if not r["answerable"]:
        return "missing fact"
    if r["ambiguous"]:
        return "ambiguous"
    if r["hops"] == 2:
        return "2-hop+explain" if r["explain"] else "2-hop"
    return "k=1" if r["k"] == 1 else ("k=2-3" if r["k"] <= 3 else "k=4-6")


sc["design"] = sc.apply(design, axis=1)
order = ["k=1", "k=2-3", "k=4-6", "missing fact", "ambiguous", "2-hop", "2-hop+explain"]
for col in ("W", "W_misleading"):
    t = sc.pivot_table(index="group", columns="design", values=col, aggfunc="mean")
    if not t.empty:
        print(t.reindex(columns=[c for c in order if c in t.columns]).round(3).to_string(), "\n")

print("3) Does the generating model's own span entropy flag filler? AUC of h_span, filler vs kept spans")
print("   (0.5 = no signal; 'within' compares spans of the same response; length AUC is a baseline):")
sp["filler"] = sp["kind"].isin(["uninformative", "duplicate", "removed"])
res = {}
for g, d in sp[sp["kind"] != "misleading"].groupby("group"):
    within = [auc(r.loc[r["filler"], "h_span"], r.loc[~r["filler"], "h_span"])
              for _, r in d.groupby("item_id") if r["filler"].any() and (~r["filler"]).any()]
    res[g] = {"filler spans": int(d["filler"].sum()), "AUC h_span": auc(d.loc[d["filler"], "h_span"],
                                                                       d.loc[~d["filler"], "h_span"]),
              "AUC h_span within": float(np.nanmean(within)) if within else float("nan"),
              "responses": len(within),
              "AUC length": auc(d.loc[d["filler"], "tokens"], d.loc[~d["filler"], "tokens"])}
print(pd.DataFrame(res).T.round(3).to_string())

print("\n4) Completeness check: requested facts the reader could read from the gold answer —")
print("   'decided' = reader answers correctly (p ≥ 0.5); 'near-full info, undecided' = ≥ 90% of the gold's")
print("   information but p < 0.5 (often a format mismatch, e.g. '23' vs '23 containers'):")
items = {j["item_id"]: j for j in map(json.loads, open(RUN / "items.jsonl"))}
resp = {(j["model"], j.get("variant", "default"), j["item_id"]): j["response"]
        for j in map(json.loads, open(RUN / "responses.jsonl"))}
recs = []
for s in map(json.loads, open(RUN / "scored.jsonl")):
    probes = [p for p in items[s["item_id"]]["probes"] if p.get("required", True)]
    for p, I, p_resp, p_gold in zip(probes, s["per_probe"], s.get("per_probe_p") or [],
                                    s.get("gold_per_probe_p") or []):
        if "not stated" in p["answers"] or p_gold < 0.5:
            continue
        f = np.log(s.get("prior_floor") or 1e-300)
        prior = max(np.log(max(p_resp, 1e-300)), f) - I  # floored log p(answer | empty message)
        I_gold = max(np.log(p_gold), f) - prior
        recs.append({"group": f"{s['model']} | {s['variant']}", "key": (s["model"], s["variant"], s["item_id"]),
                     "answer": p["answers"][0], "p": p_resp, "share_info": I / I_gold if I_gold > 0 else np.nan})
pr = pd.DataFrame(recs)
if not pr.empty:
    pr["state"] = np.select([pr["p"] >= 0.5, pr["share_info"] >= 0.9, pr["share_info"] >= 0.1],
                            ["decided", "near-full info, undecided", "partial"], "absent")
    print(pr.pivot_table(index="group", columns="state", values="p", aggfunc="size", fill_value=0)
          .pipe(lambda t: t.div(t.sum(axis=1), axis=0)).round(3).to_string())
    ex = pr[pr["state"] == "near-full info, undecided"]
    for _, r in ex.sample(min(6, len(ex)), random_state=0).iterrows():
        print(f"\n   {r['group']}: expected '{r['answer']}', p = {r['p']:.2f}")
        print("     response:", resp.get(r["key"], "")[:300].replace("\n", " / "))

print("\n5) Deleted spans by kind (count; tokens), with examples for the default prompt:")
dele = sp[sp["kind"] != "kept"]
print(pd.concat([dele.pivot_table(index="group", columns="kind", values="tokens", aggfunc="size", fill_value=0),
                 dele.pivot_table(index="group", columns="kind", values="tokens", aggfunc="sum", fill_value=0)
                 .add_suffix(" tok")], axis=1).to_string())
for kind in ("uninformative", "duplicate", "misleading"):
    for grp, g in dele[(dele["kind"] == kind) & dele["group"].str.endswith("default")].groupby("group"):
        print(f"\n  {grp} — {kind} ({len(g)} spans):")
        for t in g.sample(min(4, len(g)), random_state=0)["text"]:
            print("     ·", str(t)[:150])

print("\n6) Save this run (Colab storage is wiped when the runtime is recycled):")
archive = shutil.make_archive(str(RUN), "zip", root_dir=RUN.parent, base_dir=RUN.name)
print("   ", archive)
try:
    from google.colab import files

    files.download(archive)
except Exception:
    pass


## Second reader
Scores the same responses with another reader model and compares (~45 min on a T4).

In [ ]:
# Filler in LLM answers — score the same responses with a second reader (Colab T4, ~45 min).
# Does the benchmark depend on the reader? Run after the main cell, in the same runtime
# (or with the run folder restored). Results go to <RUN>/reader2_<LABEL2>/.

# ---- settings ----
RUN = "auto"                                  # the main cell's folder (Google Drive or /content), or a path
READER2 = "microsoft/Phi-3.5-mini-instruct"  # another model family than the main reader
LABEL2 = "phi3.5-mini"
DTYPE = "fp16"                                # T4 has no bf16
FILLER_CHECK_N = 3                             # simulated items per cell for the filler-map check (0 = skip)
FULL_RUN = True                               # False: only compare scores already there (never start ~60 min of scoring)
# ---- end settings ----

import glob
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

ENV = dict(os.environ, HF_HUB_DISABLE_PROGRESS_BARS="1", TRANSFORMERS_VERBOSITY="error",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
PROGRESS = re.compile(r"^(?:generated|signals|scored) (\d+)/(\d+)")


def run(args):
    args = [str(a) for a in args]
    shown = ["drybench"] + args[3:] if args[1:3] == ["-m", "drybench.cli"] else args
    print("$", " ".join(shown), flush=True)
    t0 = time.time()
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENV)
    for line in p.stdout:
        m = PROGRESS.match(line.strip())
        if m and int(m.group(1)) % max(10, int(m.group(2)) // 10) and m.group(1) != m.group(2):
            continue
        print("   ", line, end="", flush=True)
    code = p.wait()
    print(f"    [{time.time() - t0:.0f}s]", flush=True)
    if code:
        raise RuntimeError(f"step failed with exit code {code}: {' '.join(args)}")


def ci(x, ratio=False, n=2000):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 3:
        return "n/a"
    b = np.random.default_rng(0).choice(x, (n, len(x))).mean(axis=1)
    m, lo, hi = x.mean(), np.quantile(b, 0.025), np.quantile(b, 0.975)
    if ratio:
        return f"×{np.exp(m):.2f} [{np.exp(lo):.2f}, {np.exp(hi):.2f}]"
    return f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]"


def compare(out, r2):
    pd.set_option("display.width", 200)
    print("\n================ MAIN READER vs", LABEL2, "================")
    checks = {"main": out / "reader_check.json", LABEL2: r2 / "reader_check.json"}
    print(pd.DataFrame({k: (lambda v: {"AUC present vs absent": v["auc_present_vs_absent"],
                                       "gold unreadable": v["reader_fail_rate"],
                                       "wrong-statement detection": v.get("misleading_check_active", np.nan),
                                       "nats per fact": v["gain_present_median"]})(json.loads(p.read_text()))
                        for k, p in checks.items()}).round(3).to_string())

    rows = {}
    for k, path in (("main", out / "filler_check" / "summary.json"), (LABEL2, r2 / "filler_check" / "summary.json")):
        if path.exists():
            ws = json.loads(path.read_text())
            for key, name in (("loo", "τ=0.5"), ("loo_t1", "τ=1"), ("loo_t2", "τ=2")):
                if key in ws:
                    rows[f"{k}, {name}"] = {"no fact lost": ws[key]["safety_all_ok"],
                                            "precision": ws[key]["filler_precision"],
                                            "recall": ws[key]["filler_recall"], "MAE": ws[key]["mae"]}
    if rows:
        print("\nFiller-map check on simulated responses with known filler:")
        print(pd.DataFrame(rows).T.round(3).to_string())

    a = pd.read_csv(out / "report" / "item_table.csv")
    b = pd.read_csv(r2 / "report" / "item_table.csv")
    m = a.merge(b, on=["model", "variant", "item_id"], suffixes=("_1", "_2"))
    both = m[(m["status_1"] == "ok") & (m["status_2"] == "ok")]
    print(f"\nPer-response agreement, Spearman ({len(both)} responses scored by both; "
          "pooled / mean within model × prompt; W_ablation = W with the looser tolerance):")
    agree = {}
    for c in ("W", "W_ablation", "W_misleading", "kappa", "facts", "overhead"):
        if f"{c}_1" not in both or f"{c}_2" not in both:
            continue
        pooled = both[f"{c}_1"].corr(both[f"{c}_2"], method="spearman")
        within = np.array([d[f"{c}_1"].corr(d[f"{c}_2"], method="spearman")
                           for _, d in both.groupby(["model", "variant"])], float)
        agree[c] = {"pooled": pooled, "within": np.nanmean(within) if np.isfinite(within).any() else np.nan}
    print(pd.DataFrame(agree).T.round(3).to_string())

    sa = pd.read_csv(out / "report" / "span_table.csv")
    sb = pd.read_csv(r2 / "report" / "span_table.csv")
    s = sa.merge(sb, on=["model", "variant", "item_id", "span_idx"], suffixes=("_1", "_2"))
    w1 = s["kind_1"].isin(["uninformative", "duplicate", "removed"]).to_numpy()
    w2 = s["kind_2"].isin(["uninformative", "duplicate", "removed"]).to_numpy()
    po = float((w1 == w2).mean())
    pe = w1.mean() * w2.mean() + (1 - w1.mean()) * (1 - w2.mean())
    print(f"\nSpan-level filler map: same decision for {100 * po:.1f}% of {len(s)} spans, "
          f"Cohen's κ = {(po - pe) / (1 - pe):.2f} (filler share {w1.mean():.3f} vs {w2.mean():.3f})")

    ga = pd.read_csv(out / "report" / "groups.csv", index_col=0)
    gb = pd.read_csv(r2 / "report" / "groups.csv", index_col=0)
    cols = [c for c in ("W_mean", "W_misleading_mean", "kappa_mean", "facts_mean", "overhead_gmean")
            if c in ga and c in gb]
    print("\nPer model × prompt:")
    print(pd.concat({"main": ga[cols], LABEL2: gb[cols]}, axis=1).round(3).to_string())
    for c in ("W_mean", "overhead_gmean", "facts_mean"):
        if c in cols:
            print(f"  ranking of the {len(ga)} groups by {c}: Kendall τ = {ga[c].corr(gb[c], method='kendall'):.2f}")

    print(f"\nH3 with {LABEL2} as the reader (concise − default; overhead as a ratio):")
    piv = b.pivot_table(index=["model", "item_id"], columns="variant", values=["W", "facts", "overhead"],
                        aggfunc="first", dropna=False)
    for mdl, g in piv.groupby(level=0):
        with np.errstate(divide="ignore", invalid="ignore"):
            lr = np.log(g[("overhead", "concise")] / g[("overhead", "default")])
        print(f"  {mdl}: ΔW {ci(g[('W', 'concise')] - g[('W', 'default')])}  "
              f"Δfacts {ci(g[('facts', 'concise')] - g[('facts', 'default')])}  overhead {ci(lr, ratio=True)}")
    print(f"\nAll tables: {r2 / 'report'}")


def find_run(run, name="t4_n20_v12"):
    """RUN = "auto": the main cell's folder in this session (RUN_DIR), else the run in Google Drive,
    else the one in /content; a path is used as given."""
    if run != "auto":
        return Path(run)
    if "RUN_DIR" in globals():
        return Path(globals()["RUN_DIR"])
    name = globals().get("RUN_NAME", name)
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception:
            pass  # not in Colab, or Drive declined: look in /content
    on_drive = Path("/content/drive/MyDrive/drybench_runs") / name
    return on_drive if on_drive.exists() else Path("/content/drybench_runs") / name


def main():
    t_start = time.time()
    try:
        from google.colab import userdata

        tok = userdata.get("HF_TOKEN")
        if tok:
            ENV["HF_TOKEN"] = tok
    except Exception:
        pass  # public models do not need a token
    out = find_run(RUN)
    print("Run:", out)
    need = [out / "items.jsonl", out / "responses.jsonl", out / "signals.jsonl", out / "report" / "item_table.csv"]
    missing = [str(p) for p in need if not p.exists()]
    if missing:
        raise SystemExit("Run the main cell first; not found: " + ", ".join(missing))
    r2 = out / f"reader2_{LABEL2}"
    if not (r2 / "scored.jsonl").exists() and not FULL_RUN:
        raise SystemExit(f"No second-reader scores in {r2}: scoring every response takes ~60 min on a T4. "
                         "Set FULL_RUN = True to start it (or restore the folder from a saved run).")
    r2.mkdir(exist_ok=True)
    cli = [sys.executable, "-m", "drybench.cli"]
    version = subprocess.run([sys.executable, "-c", "import drybench; print(drybench.SCORING_VERSION)"],
                             capture_output=True, text=True, env=ENV).stdout.strip()
    main_v = json.loads((out / "versions.json").read_text()).get("scoring") if (out / "versions.json").exists() else None
    if main_v != version:
        raise SystemExit("The main run was scored by another drybench version; run the main cell first.")
    v2 = r2 / "versions.json"
    old_v = json.loads(v2.read_text()).get("scoring") if v2.exists() else (
        "older" if (r2 / "scored.jsonl").exists() else version)
    if old_v != version:  # every score, or only those the new version changed for this reader
        got = subprocess.run([str(a) for a in cli + ["prune-scores", "--dir", r2, "--items", out / "items.jsonl",
                                                     "--reader", READER2, "--from", old_v, "--json"]],
                             capture_output=True, text=True, env=ENV)
        if got.returncode:
            raise RuntimeError("prune-scores failed:\n" + got.stdout + got.stderr)
        res = json.loads(got.stdout.strip().splitlines()[-1])
        what = {"all": "every response", "missing": "the answers to items with a missing fact",
                "none": "nothing (this reader's scores did not change)"}[res["plan"]]
        print(f"Scoring changed since the last second-reader run (v{old_v} → v{version}): rescoring {what}.")
        for line in res["changes"]:
            print("    " + line)
    v2.write_text(json.dumps({"scoring": version}))
    try:
        import torch

        dtype = DTYPE if torch.cuda.is_available() else "auto"
    except ImportError:
        dtype = "auto"

    print("[1/4] reader check")
    if not (r2 / "reader_check.json").exists():
        run(cli + ["check-reader", "--items", out / "items.jsonl", "--reader-backend", "hf", "--reader", READER2,
                   "--reader-dtype", dtype, "--n", 30, "--out", r2 / "reader_check.json"])

    print("\n[2/4] filler-map check on simulated responses with known filler")
    scripts = sorted(glob.glob("/content/drybench_pkg_*/scripts/pilot_validation.py")
                     + glob.glob("drybench_pkg_*/scripts/pilot_validation.py"), key=os.path.getmtime)
    if FILLER_CHECK_N and scripts and not (r2 / "filler_check" / "summary.json").exists():
        run([sys.executable, scripts[-1], "--reader-backend", "hf", "--reader", READER2, "--reader-dtype", dtype,
             "--out-dir", r2 / "filler_check", "--n-per-cell", FILLER_CHECK_N, "--greedy-conditions", "none",
             "--no-plots", "--quiet"])

    print("\n[3/4] filler maps of the same responses")
    run(cli + ["score", "--items", out / "items.jsonl", "--responses", out / "responses.jsonl",
               "--out", r2 / "scored.jsonl", "--reader-backend", "hf", "--reader", READER2, "--reader-dtype", dtype,
               "--reader-max-batch-tokens", 4096, "--mode", "loo", "--ablation-abs-tol", 2.0])

    print("\n[4/4] analysis")
    run(cli + ["analyze", "--items", out / "items.jsonl", "--scored", r2 / "scored.jsonl",
               "--signals", out / "signals.jsonl", "--out-dir", r2 / "report", "--n-boot", 500])
    compare(out, r2)
    print(f"\nTotal time: {(time.time() - t_start) / 60:.1f} min")


main()


## Tables for the paper
CPU only, after the cells above (or after restoring a run): all tables for both readers in one Markdown file, plus a blind sample for human labels; downloads them as a zip.

In [ ]:
# Filler in LLM answers, synthetic reports — all tables for the paper from a finished run
# (CPU, ~1 min; no GPU needed). Run after the main and second-reader cells, or in a new session after
# the restore cell. Writes <RUN>/paper/ (tables.md, CSVs, a blind sample for human labels) and
# downloads it as a zip. The tables are built by the same code as the real-text tables (same columns).
# The drybench package (v1.2.0) is embedded at the bottom; nothing to install.

# ---- settings ----
RUN = "auto"                           # the main cell's folder (Google Drive or /content), or a path
N_ANNOTATE = 30                        # responses per model × prompt for human labels (0 = none)
# ---- end settings ----

import base64
import hashlib
import io
import json
import random
import shutil
import sys
import warnings
import zipfile
from pathlib import Path

import pandas as pd


def load_package():
    """Extract the embedded package next to the run and import it from there (no pip)."""
    digest = hashlib.sha1(PKG_B64.encode()).hexdigest()[:10]
    base = Path("/content") if Path("/content").exists() else Path.cwd()
    pkg = (base / f"drybench_src_{digest}").resolve()
    if not (pkg / "drybench").exists():
        zipfile.ZipFile(io.BytesIO(base64.b64decode(PKG_B64))).extractall(pkg)
    for name in [m for m in sys.modules if m == "drybench" or m.startswith("drybench.")]:
        del sys.modules[name]  # an older copy imported earlier in this session
    sys.path.insert(0, str(pkg))


def find_run(run, name="t4_n20_v12"):
    """RUN = "auto": the main cell's folder in this session (RUN_DIR), else the run in Google Drive,
    else the one in /content; a path is used as given."""
    if run != "auto":
        return Path(run)
    if "RUN_DIR" in globals():
        return Path(globals()["RUN_DIR"])
    name = globals().get("RUN_NAME", name)
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception:
            pass  # not in Colab, or Drive declined: look in /content
    on_drive = Path("/content/drive/MyDrive/drybench_runs") / name
    return on_drive if on_drive.exists() else Path("/content/drybench_runs") / name


def main():
    load_package()
    warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty bootstrap slices in tiny groups
    from drybench.report import _readers, _results, synthetic_tables
    from drybench.types import load_items

    run = find_run(RUN)
    print("Run:", run)
    if not (run / "scored.jsonl").exists():
        raise SystemExit(f"No scored run in {run}: run the main cell (or the restore cell) first.")
    out = run / "paper"
    out.mkdir(exist_ok=True)
    text = synthetic_tables(run)
    (out / "tables.md").write_text(text, encoding="utf-8")
    print(text)
    items = load_items(run / "items.jsonl")
    _, R = _readers(run, items)
    for r in R:
        _results(r["items"]).to_csv(out / f"results_{r['label']}.csv")

    # blind sample for human labels
    if N_ANNOTATE:
        rng = random.Random(0)
        by_id = {i.item_id: i for i in items}
        resp = {(j["model"], j.get("variant", "default"), j["item_id"]): j["response"]
                for j in map(json.loads, open(run / "responses.jsonl"))}
        spans = {r["label"]: r["spans"] for r in R}
        main_ok = R[0]["items"][R[0]["items"]["status"] == "ok"].copy()
        picked = []
        for g, d in main_ok.groupby("group"):
            rows = d.sample(frac=1.0, random_state=0)
            per = max(1, N_ANNOTATE // max(1, rows["design"].nunique()))
            take = rows.groupby("design").head(per)
            if len(take) < N_ANNOTATE:
                take = pd.concat([take, rows.drop(take.index).head(N_ANNOTATE - len(take))])
            picked += take.head(N_ANNOTATE)[["model", "variant", "item_id"]].values.tolist()
        rng.shuffle(picked)
        sample, key = [], []
        for n, (mdl, var, iid) in enumerate(picked, 1):
            uid = f"r{n:03d}"
            sp0 = spans[R[0]["label"]]
            rows = sp0[(sp0["model"] == mdl) & (sp0["variant"] == var) & (sp0["item_id"] == iid)].sort_values("span_idx")
            sample.append({"uid": uid, "document": by_id[iid].document, "request": by_id[iid].request,
                           "response": resp[(mdl, var, iid)],
                           "spans": [{"i": int(i), "text": t} for i, t in zip(rows["span_idx"], rows["text"])]})
            k = {"uid": uid, "model": mdl, "variant": var, "item_id": iid}
            for lab, sp_ in spans.items():
                x = sp_[(sp_["model"] == mdl) & (sp_["variant"] == var) & (sp_["item_id"] == iid)].sort_values("span_idx")
                k[f"kinds_{lab}"] = x["kind"].tolist()
            key.append(k)
        ann = out / "annotation"
        ann.mkdir(exist_ok=True)
        (ann / "sample.jsonl").write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in sample))
        (ann / "key.jsonl").write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in key))
        print(f"\nAnnotation sample: {len(sample)} responses, "
              f"{sum(len(x['spans']) for x in sample)} spans → {ann}")

    archive = shutil.make_archive(str(run / f"{run.name}_paper"), "zip", root_dir=out)
    print("\nSaved:", archive)
    try:
        from google.colab import files

        files.download(archive)
    except Exception:
        pass


PKG_B64 = "..."

main()
